# 📓 Credit Risk Feature Selection: Condition Index + VDP Analysis

This notebook helps identify multicollinear features using:
- Condition Index (CI)
- Variance Decomposition Proportions (VDP)

In [ ]:
# Step 1: Import libraries
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [ ]:
# Step 2: Define the main function
def compute_condition_index_and_vdp(X, ci_threshold=30, vdp_threshold=0.5):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    feature_names = X.columns

    corr_matrix = np.corrcoef(X_scaled, rowvar=False)
    eigenvalues, eigenvectors = np.linalg.eig(corr_matrix)
    condition_indices = np.sqrt(eigenvalues.max() / eigenvalues)

    inv_eigenvalues = 1 / eigenvalues
    phi = eigenvectors.T
    vdp_matrix = []

    for i in range(len(eigenvalues)):
        component_variance = []
        for j in range(len(feature_names)):
            loading_squared = (phi[i, j] ** 2)
            contribution = loading_squared * inv_eigenvalues[i]
            component_variance.append(contribution)
        vdp_matrix.append(component_variance)

    vdp_matrix = np.array(vdp_matrix).T
    vdp_matrix = vdp_matrix / vdp_matrix.sum(axis=1)[:, np.newaxis]

    ci_df = pd.DataFrame(vdp_matrix, columns=[f'Comp_{i+1}_CI_{round(condition_indices[i], 2)}' for i in range(len(condition_indices))])
    ci_df.insert(0, 'Feature', feature_names)

    flagged = []
    for comp_idx, ci in enumerate(condition_indices):
        if ci > ci_threshold:
            high_vdp_features = ci_df[ci_df.iloc[:, comp_idx + 1] > vdp_threshold]['Feature'].tolist()
            if len(high_vdp_features) >= 2:
                flagged.append((f'Comp_{comp_idx+1}_CI_{round(ci, 2)}', high_vdp_features))

    return condition_indices, ci_df, flagged

In [ ]:
# Step 3: Load a dataset (replace with your own dataset)
import seaborn as sns
df = sns.load_dataset("iris").drop(columns=["species"])

# Step 4: Run the CI + VDP analysis
condition_indices, vdp_table, multicollinear_groups = compute_condition_index_and_vdp(df)

# Step 5: Display results
print("📊 Condition Indices:")
print(condition_indices)

print("\n📄 Variance Decomposition Table:")
display(vdp_table)

print("\n🚨 Flagged Multicollinear Feature Groups:")
for component, features in multicollinear_groups:
    print(f"{component}: {features}")